# External classifier pre-screening on REAL RAVDESS frames

**Question this notebook answers:** which third-party FER classifiers are
*useful evaluators* of our generated videos?

Pre-screen on real (ungenerated) RAVDESS frames across **all three splits**
(train, val, test, plus a merged train+val) for cross-split consistency.
An external is a reliable evaluator only if:
  1. F1_test > ~0.5 (has headroom to differentiate model variants), AND
  2. F1_test ≈ F1_train+val (not a small-test n=96 artefact).

Internal reference: TimeSformer trained in `02_train_encoders_4emotions`
achieves macro F1 = 0.876 on real test (`04b_encoder_ceiling` cell 6).
External F1 below this is the FER↔RAVDESS domain-shift cost.

Use the ranking to pick which externals to keep in `05_external_evaluation`
/ `07a_sadtalker_loss_ablation`. If **no** external clears 0.5, that is
itself a strong methodological finding for the thesis (see closing markdown).


In [1]:
!pip install -q transformers librosa scipy scikit-learn

In [2]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import (
    accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix,
)
from tqdm import tqdm
from transformers import AutoImageProcessor, AutoModelForImageClassification

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
METADATA = "/content/processed_data/metadata.json"
OUT_DIR = Path("/content/external_screening")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EMOTIONS = ["happy", "sad", "angry", "disgust"]
EXCLUDE = {0, 1, 5, 7}
REMAP = {2: 0, 3: 1, 4: 2, 6: 3}
NUM_EMO = len(EMOTIONS)
FRAMES_PER_SAMPLE = 8   # how many frames to sample uniformly per video for classification

_NAME_ALIASES = {
    "happy":     ["happy", "happiness", "joy"],
    "sad":       ["sad", "sadness"],
    "angry":     ["angry", "anger"],
    "disgust":   ["disgust", "disgusted"],
}

print(f"Device: {DEVICE}")

Device: cuda


In [3]:
# Candidate external FER classifiers — extend this list as you discover more.
# Each entry: (display_name, hf_repo_id). Models that fail to load (404 / no
# image-classification head / etc.) are reported and skipped — see next cell.
CANDIDATES = [
    # Already used in 05/07 — kept here as the reference floor.
    ("dima806",      "dima806/facial_emotions_image_detection"),
    ("trpakov",      "trpakov/vit-face-expression"),
    # New candidates — popular ViT-based FER alternatives.
    ("motheecreator", "motheecreator/vit-Facial-Expression-Recognition"),
    ("Rajaram1996",   "Rajaram1996/FacialEmoRecog"),
    ("RickyIG",       "RickyIG/emotion_face_image_classification"),
    # SigLIP-2 family if available — newer architecture.
    ("prithivMLmods-siglip2", "prithivMLmods/Facial-Emotion-Detection-SigLIP2"),
    # Add more here as you find them — the loader is fault-tolerant.
]

In [4]:
# Load every candidate; skip ones that fail. The skipped list is shown so you
# can substitute alternates without restarting the kernel.
EXTERNALS = {}
skipped = []
for name, repo in CANDIDATES:
    try:
        proc = AutoImageProcessor.from_pretrained(repo)
        model = AutoModelForImageClassification.from_pretrained(repo).to(DEVICE).eval()
        for p in model.parameters():
            p.requires_grad = False
        id2label = model.config.id2label
        label2id_raw = {v.lower(): k for k, v in id2label.items()}
        target_ids = {}
        for our in EMOTIONS:
            picked = None
            for alias in _NAME_ALIASES.get(our, [our]):
                if alias in label2id_raw:
                    picked = label2id_raw[alias]
                    break
            if picked is None:
                raise KeyError(f"no label for '{our}' in {repo}; got {sorted(label2id_raw)}")
            target_ids[our] = picked
        EXTERNALS[name] = {
            "repo":       repo,
            "proc":       proc,
            "model":      model,
            "id2label":   id2label,
            "target_ids": target_ids,
        }
        print(f"[{name}] {repo}")
        print(f"  labels:        {id2label}")
        print(f"  target→ext id: {target_ids}")
    except Exception as exc:
        skipped.append((name, repo, str(exc).splitlines()[0]))
        print(f"[{name}] SKIPPED ({repo}): {str(exc).splitlines()[0]}")

if skipped:
    print(f"\n{len(skipped)} candidate(s) skipped — replace in CANDIDATES if needed:")
    for n, r, e in skipped:
        print(f"  - {n} ({r}): {e}")
print(f"\n{len(EXTERNALS)} candidate(s) ready for screening.")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


[dima806] dima806/facial_emotions_image_detection
  labels:        {0: 'sad', 1: 'disgust', 2: 'angry', 3: 'neutral', 4: 'fear', 5: 'surprise', 6: 'happy'}
  target→ext id: {'happy': 6, 'sad': 0, 'angry': 2, 'disgust': 1}
[trpakov] trpakov/vit-face-expression
  labels:        {0: 'angry', 1: 'disgust', 2: 'fear', 3: 'happy', 4: 'neutral', 5: 'sad', 6: 'surprise'}
  target→ext id: {'happy': 3, 'sad': 5, 'angry': 0, 'disgust': 1}


preprocessor_config.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/915 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/372M [00:00<?, ?B/s]

[prithivMLmods-siglip2] SKIPPED (prithivMLmods/Facial-Emotion-Detection-SigLIP2): "no label for 'disgust' in prithivMLmods/Facial-Emotion-Detection-SigLIP2; got ['ahegao', 'angry', 'happy', 'neutral', 'sad', 'surprise']"

1 candidate(s) skipped — replace in CANDIDATES if needed:
  - prithivMLmods-siglip2 (prithivMLmods/Facial-Emotion-Detection-SigLIP2): "no label for 'disgust' in prithivMLmods/Facial-Emotion-Detection-SigLIP2; got ['ahegao', 'angry', 'happy', 'neutral', 'sad', 'surprise']"

5 candidate(s) ready for screening.


In [5]:
# Real RAVDESS frames — load ALL splits separately so we can compare
# per-split F1 (train ≈ val ≈ test indicates external is consistent;
# big gap → eval is noisy on small test split).
with open(METADATA) as f:
    meta = json.load(f)

real_samples = {
    split_name: [s for s in meta if s["split"] == split_name and s["emotion_idx"] not in EXCLUDE]
    for split_name in ("train", "val", "test")
}

# train+val merged for "extra signal" on top of test alone.
real_samples["train+val"] = real_samples["train"] + real_samples["val"]

from collections import Counter
for split_name, ss in real_samples.items():
    counts = dict(Counter(EMOTIONS[REMAP[s["emotion_idx"]]] for s in ss))
    print(f"{split_name:10s}: n={len(ss):4d}  per-emotion={counts}")


train     : n= 544  per-emotion={'happy': 136, 'sad': 136, 'angry': 136, 'disgust': 136}
val       : n=  96  per-emotion={'happy': 24, 'sad': 24, 'angry': 24, 'disgust': 24}
test      : n=  96  per-emotion={'happy': 24, 'sad': 24, 'angry': 24, 'disgust': 24}
train+val : n= 640  per-emotion={'happy': 160, 'sad': 160, 'angry': 160, 'disgust': 160}


In [6]:
@torch.no_grad()
def predict_real(sample, ext_name):
    """Sample frames uniformly from the real video, classify with one external.
    Returns predicted target index in EMOTIONS space."""
    ext = EXTERNALS[ext_name]
    frames = np.load(sample["frames_path"])  # (T, H, W, 3) uint8
    n_total = frames.shape[0]
    if n_total < 1:
        return None
    idx = np.linspace(0, n_total - 1, min(FRAMES_PER_SAMPLE, n_total)).round().astype(int)
    sampled = frames[idx]
    pil_frames = [Image.fromarray(f) for f in sampled]
    inputs = ext["proc"](pil_frames, return_tensors="pt").to(DEVICE)
    logits = ext["model"](**inputs).logits
    probs = F.softmax(logits, dim=-1).cpu()
    mean_probs = probs.mean(dim=0)
    tgt_indices = [ext["target_ids"][e] for e in EMOTIONS]
    return int(mean_probs[tgt_indices].argmax().item())


SPLITS_TO_RUN = ["test", "val", "train", "train+val"]

# results_per_split[split][ext] = DataFrame
results_per_split = {sp: {} for sp in SPLITS_TO_RUN}
for split_name in SPLITS_TO_RUN:
    ss = real_samples[split_name]
    if not ss:
        continue
    for ext_name in EXTERNALS:
        rows = []
        for s in tqdm(ss, desc=f"{split_name} [{ext_name}]"):
            try:
                pred = predict_real(s, ext_name)
                if pred is None:
                    continue
                rows.append({
                    "sample_id":    s["sample_id"],
                    "emotion_true": REMAP[s["emotion_idx"]],
                    "ext_pred":     pred,
                })
            except Exception as exc:
                print(f"  {ext_name}/{s['sample_id']}/{split_name} failed: {exc}")
        results_per_split[split_name][ext_name] = pd.DataFrame(rows)



train+val [RickyIG]: 100%|██████████| 640/640 [00:17<00:00, 37.04it/s]


In [7]:
def _per_emotion_block(df):
    if len(df) == 0:
        return None
    y = df["emotion_true"].to_numpy()
    p = df["ext_pred"].to_numpy()
    acc = accuracy_score(y, p)
    f1m = f1_score(y, p, labels=list(range(NUM_EMO)), average="macro", zero_division=0)
    per_f1 = f1_score(y, p, labels=list(range(NUM_EMO)), average=None, zero_division=0)
    cm = confusion_matrix(y, p, labels=list(range(NUM_EMO)))
    return acc, f1m, per_f1, cm


long_rows = []
wide_rows = []
for ext_name in EXTERNALS:
    print(f"\n{'='*60}\n{ext_name} ({EXTERNALS[ext_name]['repo']})\n{'='*60}")
    wide = {"external": ext_name, "repo": EXTERNALS[ext_name]["repo"]}
    for split_name in SPLITS_TO_RUN:
        df = results_per_split[split_name].get(ext_name, pd.DataFrame())
        block = _per_emotion_block(df)
        if block is None:
            wide[f"F1_{split_name}"] = np.nan
            continue
        acc, f1m, per_f1, cm = block
        print(f"\n  [{split_name:9s}] n={len(df):4d}  acc={acc:.3f}  macro F1={f1m:.3f}")
        for i, e in enumerate(EMOTIONS):
            print(f"      {e:>8s}  F1={per_f1[i]:.3f}")
        if split_name in ("test", "train+val"):
            print(f"      confusion (rows=true, cols=pred):")
            for line in pd.DataFrame(cm, index=EMOTIONS, columns=EMOTIONS).to_string().splitlines():
                print(f"        {line}")
        wide[f"F1_{split_name}"] = f1m
        wide[f"acc_{split_name}"] = acc
        long_rows.append({
            "external": ext_name, "repo": EXTERNALS[ext_name]["repo"], "split": split_name,
            "n": len(df), "acc": acc, "macro_F1": f1m,
            **{f"F1_{EMOTIONS[i]}": per_f1[i] for i in range(NUM_EMO)},
        })
    wide_rows.append(wide)

# Long-form: one row per (external, split)
long_df = pd.DataFrame(long_rows).sort_values(["external", "split"])
long_df.to_csv(OUT_DIR / "screening_long.csv", index=False)

# Wide-form: external x split macro-F1 — easy to scan for consistency.
wide_df = pd.DataFrame(wide_rows)

# Internal ceiling reference rows (4emo encoder ceiling on real splits, from 02 / 04b).
# Internal val F1 from 02_train_encoders_4emotions cell 12: 0.7898 (test-of-train procedure on val).
# Internal real-test F1 from 04b cell 6:                       0.876.
internal_ref = {
    "external":     "INTERNAL_TimeSformer (4emo)",
    "repo":         "02_train_encoders_4emotions / 04b_encoder_ceiling",
    "F1_train":     np.nan,           # not separately measured
    "F1_val":       0.7898,           # 02 cell 12 winning encoder val F1
    "F1_test":      0.876,            # 04b cell 6 video macro F1
    "F1_train+val": np.nan,
    "acc_train":    np.nan,
    "acc_val":      np.nan,
    "acc_test":     np.nan,
    "acc_train+val": np.nan,
}
wide_df = pd.concat([wide_df, pd.DataFrame([internal_ref])], ignore_index=True)

# Sort by F1_test (the split we actually use for thesis evaluation).
wide_df = wide_df.sort_values("F1_test", ascending=False, na_position="last").reset_index(drop=True)
wide_df.to_csv(OUT_DIR / "screening_wide.csv", index=False)

print("\n=== RANKING (macro F1 by split, sorted by F1_test) ===")
cols = ["external", "F1_train", "F1_val", "F1_test", "F1_train+val"]
print(wide_df[cols].to_string(index=False))

print("\nReading guide:")
print("  - Internal TimeSformer ceiling = 0.876 on test (4emo).")
print("  - External candidates are useful if F1_test > ~0.5 AND F1_train+val ≈ F1_test")
print("    (consistency across splits → not a small-test artifact).")
print("  - Big gap between F1_train+val and F1_test → small test (n=96) is noisy;")
print("    trust the train+val number as a more reliable point estimate.")
print("  - F1_test < 0.4 → classifier is essentially noise on RAVDESS distribution;")
print("    do not use as evaluator for generated videos.")



dima806 (dima806/facial_emotions_image_detection)

  [test     ] n=  96  acc=0.396  macro F1=0.280
         happy  F1=0.608
           sad  F1=0.065
         angry  F1=0.448
       disgust  F1=0.000
      confusion (rows=true, cols=pred):
                 happy  sad  angry  disgust
        happy       24    0      0        0
        sad         13    1     10        0
        angry       10    1     13        0
        disgust      8    5     11        0

  [val      ] n=  96  acc=0.531  macro F1=0.416
         happy  F1=0.814
           sad  F1=0.235
         angry  F1=0.613
       disgust  F1=0.000

  [train    ] n= 544  acc=0.472  macro F1=0.385
         happy  F1=0.746
           sad  F1=0.340
         angry  F1=0.454
       disgust  F1=0.000

  [train+val] n= 640  acc=0.481  macro F1=0.390
         happy  F1=0.755
           sad  F1=0.324
         angry  F1=0.478
       disgust  F1=0.000
      confusion (rows=true, cols=pred):
                 happy  sad  angry  disgust
        h

## What to do with the ranking

Pick the externals whose macro F1 on real test is highest (preferably > 0.5).
Update `EXTERNAL_VIDEO_MODELS` in:
- `05_external_evaluation.ipynb` cell 7
- `07a_sadtalker_loss_ablation.ipynb` cell 12

to include only those. Re-run the relevant external-evaluation cells.

If **no** external clears 0.5 on real test, that is itself a strong
methodological finding: with current public FER models, the bottleneck for
evaluating audio-driven talking-face generation on RAVDESS is the *evaluator*,
not the *generator*. The thesis can frame the negative external result as:

> All publicly available FER classifiers tested as independent evaluators
> achieve macro F1 < 0.5 on real RAVDESS test (Table X), bounded by the
> domain shift between in-the-wild FER training data (FER2013, AffectNet)
> and studio-recorded RAVDESS. Within this evaluator-limited regime, no
> emotion-aware loss configuration achieved a transferable gain. The result
> motivates either (a) building a RAVDESS-aware external classifier from a
> non-shared family, or (b) perceptual A/B human evaluation as a complement.